# 1 · Legal-issue extraction: counts and agreement

This notebook produces the results of **Section "Legal issues extraction"** of the paper
and the corresponding appendix ("Extraction results averaged per judgment and per issue"):

| Output in this notebook | Paper table / claim |
|---|---|
| Average number of issues per judgment | Table `tab:issue_counts` |
| Pairwise agreement on issue extraction, N=20 shared judgments (pooled) | Table `tab:issue_pairwise` |
| Pairwise agreement, per-judgment averaged | Appendix Table `tab:issue_pairwise_macro` |
| LLM vs each annotator on the full N=35 sets (pooled & per-judgment) | Tables `tab:issue_n35`, `tab:issue_n35_macro` |
| Union / intersection ground truth (N=50) | Appendix Table `tab:issue_union_inter` |
| Number of hallucinated issues (4 of 78) | In-text, Section "Legal issues extraction" |

**Data.** `validation_annotator_A1.csv` / `validation_annotator_A2.csv` contain the expert
annotation of the LLM output on the 50 test judgments (one row per LLM-extracted issue;
A1 = Alessia, A2 = Piera; 20 judgments annotated by both, 15+15 by one annotator only).
`issue_alignment_A1_A2.csv` contains, for the 20 shared judgments, each annotator's own
list of issues and the manually aligned intersection of the two lists.

**Conventions.** For two issue sets X (reference system) and Y (ground truth):
P = |X∩Y|/|X|, R = |X∩Y|/|Y|, F1 = harmonic mean. *Pooled (micro)* metrics sum counts
over all judgments before computing the metric; *per-judgment (macro)* metrics compute
P/R per judgment and then average.

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

DATA = '../data'

# Expert annotations of the LLM extraction on the 50 test judgments
# (one row per LLM-extracted issue). Annotator mapping used in the paper:
#   A1 = Alessia, A2 = Piera.
df_a = pd.read_csv(f'{DATA}/validation_annotator_A1.csv')   # A1
df_p = pd.read_csv(f'{DATA}/validation_annotator_A2.csv')   # A2

sentenze_a = set(df_a['Sentenza'])
sentenze_p = set(df_p['Sentenza'])
common_sentenze = sentenze_a & sentenze_p

print(f"A1: {len(df_a)} extracted issues over {df_a['Sentenza'].nunique()} judgments")
print(f"A2: {len(df_p)} extracted issues over {df_p['Sentenza'].nunique()} judgments")
print(f"Judgments annotated by both (shared set): {len(common_sentenze)}")

# Manual alignment of the two annotators' issue lists on the 20 shared judgments
df_agreement = pd.read_csv(f'{DATA}/issue_alignment_A1_A2.csv')

A1: 54 extracted issues over 35 judgments
A2: 54 extracted issues over 35 judgments
Judgments annotated by both (shared set): 20


In [2]:
# ── Column-name constants (the CSV headers are in Italian) ──────────────────
COL_PRESENT = 'questione presente nella sentenza (TRUE/FALSE)'
COL_NOT_EXTRACTED = ("# questioni presenti NON estratte (numero minimo di questioni presenti "
                     "che in aggiunta a quelle estratte coprirebbero l'intero contenuto della sentenza)")

# ── Precision / recall / F1 helpers ──────────────────────────────────────────
def prf1(n_x, n_y, n_xy):
    """Precision (of X wrt Y), Recall (of Y found in X), F1.
    F1 is 0 when at least one of P,R is 0 (even if the other is undefined).
    F1 is NaN only when both denominators are 0."""
    p = n_xy / n_x if n_x > 0 else np.nan
    r = n_xy / n_y if n_y > 0 else np.nan
    if np.isnan(p) and np.isnan(r):
        f = np.nan
    elif np.isnan(p) or np.isnan(r):
        f = 0.0   # one side is 0 (n_xy must be 0)
    elif (p + r) == 0:
        f = 0.0
    else:
        f = 2 * p * r / (p + r)
    return p, r, f

def micro_prf1_cols(df, col_x, col_y, col_xy):
    """Pooled metrics: sum counts over all units, then compute P/R/F1."""
    return prf1(df[col_x].sum(), df[col_y].sum(), df[col_xy].sum())

def macro_prf1_cols(df, col_x, col_y, col_xy):
    # Average precision and recall over the units where each is defined, then
    # take F1 as their HARMONIC MEAN, so macro-F1 always lies between macro-P
    # and macro-R. Averaging per-unit F1 independently is unsound here:
    # one-sided units (one party lists 0 items) are forced to F1=0 yet are
    # dropped from the P or R average (their P or R is undefined/NaN), which can
    # push the mean per-unit F1 BELOW both macro-P and macro-R.
    ps, rs = [], []
    for _, row in df.iterrows():
        p, r, _ = prf1(row[col_x], row[col_y], row[col_xy])
        if not np.isnan(p): ps.append(p)
        if not np.isnan(r): rs.append(r)
    P = np.mean(ps) if ps else np.nan
    R = np.mean(rs) if rs else np.nan
    if np.isnan(P) and np.isnan(R):
        F = np.nan
    elif np.isnan(P) or np.isnan(R) or (P + R) == 0:
        F = 0.0
    else:
        F = 2 * P * R / (P + R)
    return P, R, F

fmt_pct = lambda v: f"{v:.1%}" if not np.isnan(v) else "---"


def bootstrap_micro_prf1(df, comparisons, B=10_000, seed=0):
    """95% percentile confidence intervals for the pooled P/R/F1.

    Non-parametric cluster bootstrap at the judgment level: the rows of `df`
    (one row per judgment, holding that judgment's counts) are resampled with
    replacement B times and the pooled metric is recomputed on each replicate.
    Resampling whole judgments preserves the dependence between items
    belonging to the same decision. Replicates where a denominator is 0 are
    skipped for that metric (percentiles ignore NaNs).
    """
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(df), size=(B, len(df)))
    res = {}
    for label, cx, cy, cxy in comparisons:
        X  = df[cx].to_numpy(float)[idx].sum(axis=1)
        Y  = df[cy].to_numpy(float)[idx].sum(axis=1)
        XY = df[cxy].to_numpy(float)[idx].sum(axis=1)
        with np.errstate(divide='ignore', invalid='ignore'):
            P = np.where(X > 0, XY / X, np.nan)
            R = np.where(Y > 0, XY / Y, np.nan)
            F = np.where(X + Y > 0, 2 * XY / (X + Y), np.nan)
        res[label] = {m: (np.nanpercentile(v, 2.5), np.nanpercentile(v, 97.5))
                      for m, v in [('P', P), ('R', R), ('F1', F)]}
    return res


def ci_table(point_rows, ci, order=('P', 'R', 'F1')):
    """Format 'point [lo, hi]' cells from point estimates and bootstrap CIs."""
    rows = []
    for label, pt in point_rows:
        row = {'Comparison': label}
        for m in order:
            lo, hi = ci[label][m]
            row[m] = f"{pt[m]*100:.1f} [{lo*100:.1f}, {hi*100:.1f}]"
        rows.append(row)
    return pd.DataFrame(rows).set_index('Comparison')

## Average number of issues per judgment — Table `tab:issue_counts`

For each annotator, the number of issues they identified in a judgment is the number of
LLM-extracted issues they marked as present **plus** the number of issues they considered
present but not extracted by the LLM. For the LLM it is simply the number of extracted
issues. (This reproduces the counts of `tab:issue_counts`: totals in parentheses.)

In [3]:
def annotator_issue_totals(df_ann):
    """Per-judgment number of issues identified by the annotator:
    LLM-extracted issues marked present + present-but-not-extracted issues."""
    n_present = df_ann.groupby('Sentenza')[COL_PRESENT].sum()
    n_not_extracted = df_ann.groupby('Sentenza')[COL_NOT_EXTRACTED].max().fillna(0)
    return n_present + n_not_extracted

# LLM issue counts over all 50 judgments (the LLM extraction is identical in the
# two annotators' files for the shared judgments)
llm_counts = pd.concat([
    df_a.groupby('Sentenza').size(),
    df_p[~df_p['Sentenza'].isin(sentenze_a)].groupby('Sentenza').size(),
])

rows = []
for label, df_ann in [('A1', df_a), ('A2', df_p)]:
    tot = annotator_issue_totals(df_ann)
    tot_sh = tot[tot.index.isin(common_sentenze)]
    rows.append({'': label,
                 'Shared (N=20)': f'{tot_sh.mean():.2f} ({int(tot_sh.sum())})',
                 'Full set':      f'{tot.mean():.2f} ({int(tot.sum())})  N={len(tot)}'})
llm_sh = llm_counts[llm_counts.index.isin(common_sentenze)]
rows.append({'': 'LLM',
             'Shared (N=20)': f'{llm_sh.mean():.2f} ({int(llm_sh.sum())})',
             'Full set':      f'{llm_counts.mean():.2f} ({int(llm_counts.sum())})  N={len(llm_counts)}'})

print('Average number of issues identified per judgment (totals in parentheses)')
print(pd.DataFrame(rows).set_index('').to_string())

Average number of issues identified per judgment (totals in parentheses)
    Shared (N=20)         Full set
                                  
A1      2.35 (47)  2.09 (73)  N=35
A2      1.85 (37)  1.77 (62)  N=35
LLM     1.50 (30)  1.56 (78)  N=50


## Pairwise agreement on issue extraction (N=20 shared judgments) — Tables `tab:issue_pairwise`, `tab:issue_pairwise_macro`

We compare issue extraction across the three "annotators" — A1, A2, and the LLM — on the
20 shared judgments. Inter-annotator overlap (|A1 ∩ A2|) comes from the manual alignment
in `issue_alignment_A1_A2.csv`; the LLM-vs-annotator overlap is the number of LLM issues
marked as present by that annotator.

In [4]:
# ── Per-judgment counts ───────────────────────────────────────────────────────
records = []
for _, ag_row in df_agreement.iterrows():
    sent = ag_row['Sentenza']
    n_alessia   = ag_row['# questioni Alessia']     # A1's own issue count
    n_piera     = ag_row['# questioni Piera']       # A2's own issue count
    n_inter_ap  = ag_row['# questioni intersezione']

    a_rows = df_a[df_a['Sentenza'] == sent]
    p_rows = df_p[df_p['Sentenza'] == sent]
    n_llm        = len(a_rows)
    n_llm_true_a = int(a_rows[COL_PRESENT].sum())
    n_llm_true_p = int(p_rows[COL_PRESENT].sum())

    records.append({
        'Sentenza': sent,
        'n_a1': n_alessia, 'n_a2': n_piera, 'n_llm': n_llm,
        'inter_a1a2': n_inter_ap,
        'inter_llm_a1': n_llm_true_a,
        'inter_llm_a2': n_llm_true_p,
    })

df_pairs = pd.DataFrame(records)
print(f"Shared judgments: {len(df_pairs)}")

Shared judgments: 20


In [5]:
pairs_q = [
    ('A1 | A2',  'n_a1',  'n_a2',  'inter_a1a2'),
    ('A2 | A1',  'n_a2',  'n_a1',  'inter_a1a2'),
    ('LLM | A1', 'n_llm', 'n_a1',  'inter_llm_a1'),
    ('LLM | A2', 'n_llm', 'n_a2',  'inter_llm_a2'),
]

rows_mi, rows_ma = [], []
for label, cx, cy, cxy in pairs_q:
    p, r, f = micro_prf1_cols(df_pairs, cx, cy, cxy)
    rows_mi.append({'Comparison': label, 'Precision': p, 'Recall': r, 'F1': f})
    p, r, f = macro_prf1_cols(df_pairs, cx, cy, cxy)
    rows_ma.append({'Comparison': label, 'Precision': p, 'Recall': r, 'F1': f})

print("POOLED (micro) — Table tab:issue_pairwise:")
print(pd.DataFrame(rows_mi).set_index('Comparison').map(fmt_pct).to_string())
print()
print("PER-JUDGMENT (macro) — Table tab:issue_pairwise_macro:")
print(pd.DataFrame(rows_ma).set_index('Comparison').map(fmt_pct).to_string())

POOLED (micro) — Table tab:issue_pairwise:
           Precision  Recall     F1
Comparison                         
A1 | A2        78.7%  100.0%  88.1%
A2 | A1       100.0%   78.7%  88.1%
LLM | A1       93.3%   59.6%  72.7%
LLM | A2       93.3%   75.7%  83.6%

PER-JUDGMENT (macro) — Table tab:issue_pairwise_macro:
           Precision  Recall     F1
Comparison                         
A1 | A2        91.3%  100.0%  95.5%
A2 | A1       100.0%   91.3%  95.5%
LLM | A1       95.0%   82.7%  88.4%
LLM | A2       95.0%   90.0%  92.4%


### Bootstrap confidence intervals (pooled scores, N=20)

95% percentile intervals from a cluster bootstrap at the judgment level
(B=10,000 resamples of the 20 shared judgments), as reported in brackets in the
paper's Table `tab:issue_pairwise`.

In [6]:
ci = bootstrap_micro_prf1(df_pairs, pairs_q)
points = [(label, dict(zip(['P', 'R', 'F1'], micro_prf1_cols(df_pairs, cx, cy, cxy))))
          for label, cx, cy, cxy in pairs_q]
print("Pooled issue-extraction scores with 95% bootstrap CIs (N=20):")
print(ci_table(points, ci).to_string())

Pooled issue-extraction scores with 95% bootstrap CIs (N=20):
                               P                     R                  F1
Comparison                                                                
A1 | A2        78.7 [65.6, 95.7]  100.0 [100.0, 100.0]   88.1 [79.2, 97.8]
A2 | A1     100.0 [100.0, 100.0]     78.7 [65.6, 95.7]   88.1 [79.2, 97.8]
LLM | A1      93.3 [83.9, 100.0]     59.6 [42.0, 84.8]   72.7 [56.4, 91.2]
LLM | A2      93.3 [83.9, 100.0]    75.7 [55.8, 100.0]  83.6 [67.5, 100.0]


## LLM vs each annotator on the full N=35 sets — Tables `tab:issue_n35`, `tab:issue_n35_macro`

Each annotator covers 35 judgments (20 shared + 15 own). The annotator's reference set per
judgment is: LLM issues marked present + issues present but not extracted.

In [7]:
def issue_counts_per_judgment(df_ann):
    """One row per judgment: LLM issue count, annotator issue count, matches."""
    records = []
    for sent, grp in df_ann.groupby('Sentenza'):
        n_llm = len(grp)
        n_true = int(grp[COL_PRESENT].sum())
        n_not_extr = grp[COL_NOT_EXTRACTED].max()
        n_not_extr = 0 if pd.isna(n_not_extr) else int(n_not_extr)
        records.append({'n_llm': n_llm, 'n_annotator': n_true + n_not_extr, 'n_match': n_true})
    return pd.DataFrame(records)

def llm_vs_annotator(df_r):
    p_mi, r_mi, f_mi = prf1(df_r['n_llm'].sum(), df_r['n_annotator'].sum(), df_r['n_match'].sum())
    p_ma, r_ma, f_ma = macro_prf1_cols(df_r, 'n_llm', 'n_annotator', 'n_match')
    return {
        'Precision (pooled)': p_mi, 'Recall (pooled)': r_mi, 'F1 (pooled)': f_mi,
        'Precision (per-judgment)': p_ma, 'Recall (per-judgment)': r_ma, 'F1 (per-judgment)': f_ma,
    }

df_r_a, df_r_p = issue_counts_per_judgment(df_a), issue_counts_per_judgment(df_p)
table_n35 = pd.DataFrame({
    'LLM | A1 (N=35)': llm_vs_annotator(df_r_a),
    'LLM | A2 (N=35)': llm_vs_annotator(df_r_p),
})
print("LLM issue extraction vs each annotator's full set — Tables tab:issue_n35 (pooled) and tab:issue_n35_macro:")
print(table_n35.map(fmt_pct).to_string())

LLM issue extraction vs each annotator's full set — Tables tab:issue_n35 (pooled) and tab:issue_n35_macro:
                         LLM | A1 (N=35) LLM | A2 (N=35)
Precision (pooled)                 94.4%           94.4%
Recall (pooled)                    69.9%           82.3%
F1 (pooled)                        80.3%           87.9%
Precision (per-judgment)           95.7%           95.7%
Recall (per-judgment)              86.3%           92.4%
F1 (per-judgment)                  90.8%           94.0%


In [8]:
# 95% bootstrap CIs for the pooled N=35 scores (Table tab:issue_n35, brackets)
comp = [('LLM | A1 (N=35)', 'n_llm', 'n_annotator', 'n_match')]
for tag, df_r in [('LLM | A1 (N=35)', df_r_a), ('LLM | A2 (N=35)', df_r_p)]:
    ci = bootstrap_micro_prf1(df_r, [(tag, 'n_llm', 'n_annotator', 'n_match')])
    pt = dict(zip(['P', 'R', 'F1'], prf1(df_r['n_llm'].sum(), df_r['n_annotator'].sum(), df_r['n_match'].sum())))
    print(ci_table([(tag, pt)], ci).to_string())

                                  P                  R                 F1
Comparison                                                               
LLM | A1 (N=35)  94.4 [87.9, 100.0]  69.9 [54.2, 87.5]  80.3 [67.6, 92.2]
                                  P                  R                 F1
Comparison                                                               
LLM | A2 (N=35)  94.4 [88.0, 100.0]  82.3 [67.9, 96.2]  87.9 [77.1, 97.1]


## Union / intersection ground truth (N=50) — Appendix Table `tab:issue_union_inter`

For the 20 shared judgments the ground truth combines the two annotators:
**union** = issue identified by *either* annotator; **intersection** = issue identified by
*both*. For the 30 single-annotator judgments the ground truth is that annotator's set.

In [9]:
only_a = sentenze_a - common_sentenze
only_p = sentenze_p - common_sentenze
records_50 = []

for sent in only_a:
    grp = df_a[df_a['Sentenza'] == sent]
    n_llm = len(grp); n_true = int(grp[COL_PRESENT].sum())
    n_ne = grp[COL_NOT_EXTRACTED].max(); n_ne = 0 if pd.isna(n_ne) else int(n_ne)
    n_ann = n_true + n_ne
    records_50.append({'n_llm': n_llm, 'n_ref_union': n_ann, 'n_ref_inter': n_ann,
                       'n_match_union': n_true, 'n_match_inter': n_true})

for sent in only_p:
    grp = df_p[df_p['Sentenza'] == sent]
    n_llm = len(grp); n_true = int(grp[COL_PRESENT].sum())
    n_ne = grp[COL_NOT_EXTRACTED].max(); n_ne = 0 if pd.isna(n_ne) else int(n_ne)
    n_ann = n_true + n_ne
    records_50.append({'n_llm': n_llm, 'n_ref_union': n_ann, 'n_ref_inter': n_ann,
                       'n_match_union': n_true, 'n_match_inter': n_true})

for _, ag_row in df_agreement.iterrows():
    sent = ag_row['Sentenza']
    n_alessia = ag_row['# questioni Alessia']; n_piera = ag_row['# questioni Piera']
    n_inter_ap = ag_row['# questioni intersezione']
    n_union_ap = n_alessia + n_piera - n_inter_ap
    a_rows = df_a[df_a['Sentenza'] == sent]; p_rows = df_p[df_p['Sentenza'] == sent]
    n_llm = len(a_rows)
    a_present = a_rows.set_index('Questioni estratte')[COL_PRESENT]
    p_present = p_rows.set_index('Questioni estratte')[COL_PRESENT]
    n_match_union = int((a_present | p_present).sum())
    n_match_inter = int((a_present & p_present).sum())
    records_50.append({'n_llm': n_llm, 'n_ref_union': n_union_ap, 'n_ref_inter': n_inter_ap,
                       'n_match_union': n_match_union, 'n_match_inter': n_match_inter})

df_50 = pd.DataFrame(records_50)
assert len(df_50) == 50

table_ui = pd.DataFrame({
    'Union (N=50)': {
        'Precision (pooled)':       micro_prf1_cols(df_50, 'n_llm', 'n_ref_union', 'n_match_union')[0],
        'Recall (pooled)':          micro_prf1_cols(df_50, 'n_llm', 'n_ref_union', 'n_match_union')[1],
        'Precision (per-judgment)': macro_prf1_cols(df_50, 'n_llm', 'n_ref_union', 'n_match_union')[0],
        'Recall (per-judgment)':    macro_prf1_cols(df_50, 'n_llm', 'n_ref_union', 'n_match_union')[1],
    },
    'Intersection (N=50)': {
        'Precision (pooled)':       micro_prf1_cols(df_50, 'n_llm', 'n_ref_inter', 'n_match_inter')[0],
        'Recall (pooled)':          micro_prf1_cols(df_50, 'n_llm', 'n_ref_inter', 'n_match_inter')[1],
        'Precision (per-judgment)': macro_prf1_cols(df_50, 'n_llm', 'n_ref_inter', 'n_match_inter')[0],
        'Recall (per-judgment)':    macro_prf1_cols(df_50, 'n_llm', 'n_ref_inter', 'n_match_inter')[1],
    },
})

print("LLM issue extraction with union/intersection ground truth — Table tab:issue_union_inter:")
print(table_ui.map(fmt_pct).to_string())

LLM issue extraction with union/intersection ground truth — Table tab:issue_union_inter:
                         Union (N=50) Intersection (N=50)
Precision (pooled)              94.9%               94.9%
Recall (pooled)                 75.5%               84.1%
Precision (per-judgment)        96.0%               96.0%
Recall (per-judgment)           89.1%               92.0%


## Hallucinated issues (in-text numbers)

The paper reports that of the 78 LLM-extracted issues, only **4** were marked as absent by
either annotator (2 of them on the shared set, confirmed independently by both annotators);
these 4 issues are excluded from all downstream analyses.

In [10]:
df_all = pd.concat([df_a.assign(annotator='A1'), df_p.assign(annotator='A2')], ignore_index=True)
present_joint = df_all.groupby(['Sentenza', 'Questioni estratte'])[COL_PRESENT].all()
present_any   = df_all.groupby(['Sentenza', 'Questioni estratte'])[COL_PRESENT].any()

n_issues = len(present_joint)
absent = present_joint[~present_joint]
absent_shared = absent[absent.index.get_level_values('Sentenza').isin(common_sentenze)]

print(f"LLM-extracted issues over the 50 judgments: {n_issues}")
print(f"Marked absent by at least one annotator:   {len(absent)}")
print(f"  of which on the 20 shared judgments:     {len(absent_shared)}")
print(f"  shared absences confirmed by BOTH annotators: "
      f"{int((~present_any.loc[absent_shared.index]).sum())} of {len(absent_shared)}")

LLM-extracted issues over the 50 judgments: 78
Marked absent by at least one annotator:   4
  of which on the 20 shared judgments:     2
  shared absences confirmed by BOTH annotators: 2 of 2
